# manasight 05c: fine-tuning CLIP

**Question:** if CLIP itself learns from the cards, does it get past what its frozen features could give?

Notebooks 05 and 05b trained classifiers on top of a frozen CLIP and both stopped at about 45% on color.
A bigger classifier did not help: the limit was in CLIP's 512 numbers, not in the reader.
Here CLIP's own weights are trained, so those 512 numbers can change to suit the question.

```
picture -> CLIP image network -> 512 numbers -> last layer -> 7 classes
           \__________ all of it is trained here __________/
```

This is the same two-phase recipe as in transfer learning generally:
1. **Phase 1:** CLIP frozen, train only the new last layer. (Otherwise a random last layer would send large,
   meaningless corrections back into CLIP and damage what it already knows.)
2. **Phase 2:** unfreeze everything and train with a very small learning rate.

**This notebook needs a GPU.** Every picture goes through the full network on every epoch, about 88 million weights.
On a laptop CPU one epoch takes many hours; on a Colab T4 GPU a few minutes.
In VS Code: Select Kernel -> Colab -> a GPU runtime. The first cell then fetches the code and the art onto that machine.

Sections:
1. Setup (code, GPU, art)
2. Data loaders
3. The model
4. Phase 1: train the last layer
5. Phase 2: fine-tune everything
6. Test results
7. Summary

In [ ]:
import sys
from pathlib import Path

# On Colab the notebook runs on Google's machine, which does not have our files.
# Fetch the code from GitHub (the art is downloaded in the next step).
ON_COLAB = Path("/content").exists()
if ON_COLAB and not Path("/content/manasight").exists():
    !git clone https://github.com/codecustard/manasight.git /content/manasight
if ON_COLAB:
    !pip install -q open_clip_torch
    ROOT = Path("/content/manasight")
else:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
print("project folder:", ROOT)

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

from manasight.data import COLOR_CLASSES, COLOR_NAMES, TYPE_CLASSES, load_cards, download_art
from manasight.clip import load_clip
from manasight.finetune import ArtDataset, ClipClassifier, train_transform, set_clip_trainable, predict
from manasight.model import count_weights
from manasight.engine import train
from manasight.evaluate import score

plt.rcParams["figure.dpi"] = 72

TASK = "color"        # "color" or "type": which label to train for
QUICK_TEST = False    # True: 200 pictures and 1 epoch per phase, to check that everything runs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Fine-tuning on a CPU takes many hours; use QUICK_TEST = True or switch to a GPU runtime.")

## 1. Setup: the art

Only the pictures of the main split are needed (train, val, test). On a fresh Colab machine this downloads them from
Scryfall with the same function as notebook 01 (about 20 to 40 minutes; safe to stop and run again).
On a machine that already has `data/art`, it finds everything in place and does nothing.

In [ ]:
cards = load_cards()
cards = cards[cards.split.isin(["train", "val", "test"])].reset_index(drop=True)
if QUICK_TEST:
    cards = pd.concat([cards[cards.split == "train"].sample(120, random_state=0),
                       cards[cards.split == "val"].sample(40, random_state=0),
                       cards[cards.split == "test"].sample(40, random_state=0)]).reset_index(drop=True)

for attempt in range(3):
    failed = download_art(cards, workers=16)
    if len(failed) == 0:
        break
print(cards.split.value_counts().to_string())

## 2. Data loaders

- **Training pictures** get a random crop and a random left-right flip, different every epoch (decided in notebook 02).
- **Validation and test pictures** get CLIP's own fixed preparation: resize to 224 and center crop.

In [ ]:
clip_model, eval_transform, tokenizer = load_clip()

classes = COLOR_CLASSES if TASK == "color" else TYPE_CLASSES
class_number = {name: i for i, name in enumerate(classes)}
labels = np.array([class_number[name] for name in cards[TASK]])

is_train = (cards.split == "train").to_numpy()
is_val = (cards.split == "val").to_numpy()
is_test = (cards.split == "test").to_numpy()

train_data = ArtDataset(cards.path[is_train], labels[is_train], train_transform())
val_data = ArtDataset(cards.path[is_val], labels[is_val], eval_transform)
test_data = ArtDataset(cards.path[is_test], labels[is_test], eval_transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=2)
print(f"task: {TASK}   train {len(train_data):,}   val {len(val_data):,}   test {len(test_data):,}")

## 3. The model

`ClipClassifier` (in `src/manasight/finetune.py`) is CLIP's image network with a new last layer:
512 numbers in, one score per class out. The text half of CLIP is not used here.

In [ ]:
model = ClipClassifier(clip_model.visual, classes=len(classes))
print(f"weights in CLIP's image network: {count_weights(model.visual):,}")
print(f"weights in the new last layer:   {count_weights(model.head):,}")
loss_fn = nn.CrossEntropyLoss()

## 4. Phase 1: train the last layer

CLIP is frozen. Only the new last layer learns (Adam, learning rate 0.001). This is a linear probe again, now trained
on augmented pictures, so its result should land near notebook 05.

In [ ]:
set_clip_trainable(model, False)
optimizer = torch.optim.Adam(model.head.parameters(), lr=0.001)

start = time.time()
history_1 = train(model, train_loader, val_loader, optimizer, loss_fn,
                  epochs=1 if QUICK_TEST else 3, device=device)
print(f"phase 1 took {(time.time() - start) / 60:.1f} minutes")

## 5. Phase 2: fine-tune everything

Now CLIP's weights are trained too.

- **Two learning rates:** 0.00001 for CLIP (a hundred times smaller than usual, so it adjusts what it knows instead of
  overwriting it) and 0.0001 for the last layer.
- **AdamW with weight decay 0.05:** holds the weights back, against memorizing.
- **Learning-rate schedule:** halve both rates when the validation loss stops improving for 1 epoch.
- **Early stopping:** stop after 2 epochs without improvement, keep the best weights. At most 8 epochs.

In [ ]:
set_clip_trainable(model, True)
optimizer = torch.optim.AdamW([{"params": model.visual.parameters(), "lr": 0.00001},
                               {"params": model.head.parameters(), "lr": 0.0001}], weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=1)

start = time.time()
history_2 = train(model, train_loader, val_loader, optimizer, loss_fn,
                  epochs=1 if QUICK_TEST else 8, device=device, scheduler=scheduler, patience=2)
print(f"phase 2 took {(time.time() - start) / 60:.1f} minutes")

In [ ]:
train_loss = history_1["train_loss"] + history_2["train_loss"]
val_loss = history_1["val_loss"] + history_2["val_loss"]
train_accuracy = history_1["train_acc"] + history_2["train_acc"]
val_accuracy = history_1["val_acc"] + history_2["val_acc"]
epochs = range(1, len(train_loss) + 1)
phase_1_end = len(history_1["train_loss"]) + 0.5

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].plot(epochs, train_loss, marker="o", label="train")
axes[0].plot(epochs, val_loss, marker="o", label="val")
axes[0].set_title("loss per epoch")
axes[1].plot(epochs, train_accuracy, marker="o", label="train")
axes[1].plot(epochs, val_accuracy, marker="o", label="val")
axes[1].set_title("accuracy per epoch")
for ax in axes:
    ax.axvline(phase_1_end, color="grey", linestyle="--")
    ax.set_xlabel("epoch (dashed line: CLIP unfrozen)")
    ax.legend()
plt.tight_layout()
plt.show()

**Finding:** _written after the first full run._

## 6. Test results

The 10 held-out sets.

In [ ]:
test_cards = cards[is_test]
probabilities = predict(model, test_loader, device)
result = score(test_cards[TASK], probabilities, classes)

if TASK == "color":
    earlier = [{"model": "always Multicolor", "accuracy": 0.202, "macro-F1": 0.048},
               {"model": "color histogram (03)", "accuracy": 0.340, "macro-F1": 0.304, "top-2": 0.542},
               {"model": "CLIP linear probe (05)", "accuracy": 0.446, "macro-F1": 0.442, "top-2": 0.675},
               {"model": "CLIP small network (05b)", "accuracy": 0.448, "macro-F1": 0.439, "top-2": 0.668}]
else:
    earlier = [{"model": "always Creature", "accuracy": 0.499, "macro-F1": 0.095},
               {"model": "CLIP linear probe (05)", "accuracy": 0.578, "macro-F1": 0.396, "top-2": 0.742},
               {"model": "CLIP small network (05b)", "accuracy": 0.585, "macro-F1": 0.421, "top-2": 0.739}]
results = pd.DataFrame(earlier + [{"model": "CLIP fine-tuned", **result}]).set_index("model")
results.round(3)

In [ ]:
predicted = np.array(classes)[probabilities.argmax(axis=1)]
matrix = confusion_matrix(test_cards[TASK], predicted, labels=classes, normalize="true")
names = [COLOR_NAMES.get(name, name) for name in classes]

plt.figure(figsize=(6.5, 5.5))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=0.8)
plt.xticks(range(len(classes)), names, rotation=45, ha="right")
plt.yticks(range(len(classes)), names)
for row in range(len(classes)):
    for column in range(len(classes)):
        plt.text(column, row, f"{matrix[row, column]:.2f}", ha="center", va="center", fontsize=9)
plt.xlabel("predicted")
plt.ylabel("true")
plt.title(f"CLIP fine-tuned, {TASK} (rows add up to 1)")
plt.colorbar()
plt.tight_layout()
plt.show()

print("share of each class found (recall):")
print(pd.Series(matrix.diagonal(), index=names).round(2).to_string())

print("\naccuracy by frame era:")
for era in ["1993", "1997", "2003", "2015", "2024+"]:
    in_era = (test_cards.era == era).to_numpy()
    if in_era.sum() > 0:
        print(f"{era:6s} {score(test_cards[TASK][in_era], probabilities[in_era], classes)['accuracy']:.3f}")

In [ ]:
# keep the trained weights and the test predictions (on Colab these stay on Google's machine until downloaded)
models_folder = ROOT / "models"
models_folder.mkdir(exist_ok=True)
if not QUICK_TEST:
    torch.save(model.state_dict(), models_folder / f"clip_finetuned_{TASK}.pt")
    np.save(models_folder / f"clip_finetuned_{TASK}_test_probabilities.npy", probabilities)
    print("saved to", models_folder)

**Finding:** _written after the first full run._

## 7. Summary

_Written after the first full run._